## ReAct

`ReAct` significa **Reasoning and Acting**.

Diferentemente de um `Predict` ou `ChainOfThought`, o módulo `ReAct`
permite que o modelo interaja com ferramentas externas durante a resolução
de uma tarefa.

O funcionamento geral é:

1. O modelo recebe a pergunta.
2. Analisa o que precisa fazer.
3. Escolhe uma ferramenta.
4. Define os argumentos da ferramenta.
5. Executa a ferramenta.
6. Observa o resultado.
7. Decide se precisa utilizar outra ferramenta.
8. Quando possui informações suficientes, produz a resposta final.

Neste exemplo disponibilizaremos duas ferramentas ao agente:

- `consultar_acao`: consulta informações financeiras utilizando `yfinance`.
- `obter_piada_chuck_norris`: consulta a API `chucknorris.io`.

O ponto importante é que **não escolhemos manualmente qual ferramenta será
executada**. O próprio agente ReAct decide isso com base na pergunta.

In [1]:
import os
from dotenv import load_dotenv
import dspy
import json
import requests
import yfinance as yf

load_dotenv()

True

## Setup - Configuração do Modelo

Carregamos as variáveis de ambiente do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [2]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Configura o modelo padrão utilizado pelo DSPy
dspy.configure(lm=lm)

## Tool - Consulta de Ações com yfinance

A primeira ferramenta permite consultar a última cotação diária disponível
de uma ação.

O parâmetro `ticker` representa o código utilizado pelo Yahoo Finance.

Exemplos:

- `AAPL` → Apple
- `MSFT` → Microsoft
- `NVDA` → NVIDIA
- `PETR4.SA` → Petrobras
- `VALE3.SA` → Vale

Para ações brasileiras negociadas na B3, normalmente utilizamos o sufixo `.SA`.

A função também calcula a variação percentual entre os dois últimos
fechamentos disponíveis.

In [3]:
def consultar_acao(ticker: str) -> str:
    """
    Consulta a última cotação diária disponível de uma ação ou ativo
    utilizando yfinance.

    Args:
        ticker: Símbolo do ativo no Yahoo Finance.
                Exemplos:
                AAPL
                MSFT
                NVDA
                PETR4.SA
                VALE3.SA

    Returns:
        String JSON contendo:
        - ticker
        - data da última cotação
        - último fechamento
        - fechamento anterior
        - variação percentual
    """

    try:
        # Remove espaços e padroniza o ticker em letras maiúsculas
        ticker = ticker.strip().upper()

        # Cria um objeto representando o ativo no Yahoo Finance
        ativo = yf.Ticker(ticker)

        # Obtém os últimos dias de negociação
        historico = ativo.history(
            period="5d",       # Últimos 5 dias
            interval="1d",     # Dados diários
            auto_adjust=False  # Mantém preços sem ajuste automático
        )

        # Remove linhas nas quais o fechamento não está disponível
        historico = historico.dropna(subset=["Close"])

        # Verifica se algum dado foi encontrado
        if historico.empty:
            return json.dumps(
                {
                    "erro": f"Nenhum dado encontrado para o ticker {ticker}."
                },
                ensure_ascii=False
            )

        # Último preço de fechamento disponível
        preco_atual = float(historico["Close"].iloc[-1])

        # Data correspondente ao último fechamento
        data_atual = historico.index[-1].strftime("%Y-%m-%d")

        # Calcula a variação em relação ao fechamento anterior
        if len(historico) >= 2:

            preco_anterior = float(
                historico["Close"].iloc[-2]
            )

            variacao = (
                (preco_atual - preco_anterior)
                / preco_anterior
            ) * 100

        else:
            preco_anterior = None
            variacao = None

        # Estrutura a resposta que será devolvida ao agente
        resultado = {
            "ticker": ticker,
            "data": data_atual,
            "ultimo_fechamento": round(preco_atual, 2),
            "fechamento_anterior": (
                round(preco_anterior, 2)
                if preco_anterior is not None
                else None
            ),
            "variacao_percentual": (
                round(variacao, 2)
                if variacao is not None
                else None
            )
        }

        # ReAct recebe uma representação textual do resultado
        return json.dumps(
            resultado,
            ensure_ascii=False
        )

    except Exception as erro:

        return json.dumps(
            {
                "erro": f"Falha ao consultar {ticker}: {str(erro)}"
            },
            ensure_ascii=False
        )

## Testando a Tool do Yahoo Finance

Antes de fornecer a ferramenta ao agente ReAct, podemos executá-la
diretamente para verificar seu funcionamento.

In [4]:
consultar_acao("NVDA")

'{"ticker": "NVDA", "data": "2026-09-04", "ultimo_fechamento": 230.36, "fechamento_anterior": 228.45, "variacao_percentual": 0.84}'

In [5]:
consultar_acao("PETR4.SA")

'{"ticker": "PETR4.SA", "data": "2026-09-04", "ultimo_fechamento": 47.11, "fechamento_anterior": 47.56, "variacao_percentual": -0.95}'

## Tool - Chuck Norris API

A segunda ferramenta consulta a API pública `chucknorris.io`.

A API oferece piadas/fatos satíricos de Chuck Norris.

É possível:

- obter uma piada completamente aleatória;
- selecionar uma categoria específica.

Quando nenhuma categoria é informada, utilizamos:

    /jokes/random

Quando uma categoria é informada:

    /jokes/random?category=categoria

Antes de realizar a consulta por categoria, nossa função também verifica
quais categorias são válidas utilizando:

    /jokes/categories

In [8]:
def obter_piada_chuck_norris(categoria: str = "") -> str:
    """
    Obtém uma piada/fato satírico de Chuck Norris utilizando
    a API pública chucknorris.io.

    Args:
        categoria: Categoria opcional da piada.

                   Exemplos comuns:
                   dev
                   movie
                   food
                   science

                   Se for uma string vazia, uma piada aleatória
                   será retornada.

    Returns:
        Texto da piada retornada pela API.
    """

    try:

        # Endpoint utilizado para obter uma piada
        url = "https://api.chucknorris.io/jokes/random"

        # Parâmetros enviados na requisição
        params = {}

        # Normaliza a categoria informada
        categoria = categoria.strip().lower()

        # Se uma categoria foi solicitada
        if categoria:

            # Consulta primeiro as categorias disponíveis na API
            categorias_response = requests.get(
                "https://api.chucknorris.io/jokes/categories",
                timeout=10
            )

            # Gera exceção caso a requisição HTTP tenha falhado
            categorias_response.raise_for_status()

            # Converte a resposta JSON para uma lista Python
            categorias = categorias_response.json()

            # Verifica se a categoria solicitada existe
            if categoria not in categorias:

                return (
                    f"Categoria '{categoria}' não encontrada. "
                    f"Categorias disponíveis: {', '.join(categorias)}"
                )

            # Adiciona a categoria à chamada da API
            params["category"] = categoria

        # Realiza a requisição para obter a piada
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        # Gera exceção caso a API retorne um erro HTTP
        response.raise_for_status()

        # Converte o JSON retornado pela API
        dados = response.json()

        # O campo "value" contém o texto da piada
        return dados["value"]

    except requests.RequestException as erro:

        return (
            "Erro ao acessar a API Chuck Norris: "
            f"{str(erro)}"
        )

## Testando a Tool Chuck Norris

Podemos verificar a ferramenta diretamente antes de disponibilizá-la
ao agente.

In [10]:
obter_piada_chuck_norris()

'Chuck Norris can get free internet access, text messages and smart phone app functionality from train station pay phones.'

In [11]:
obter_piada_chuck_norris("dev")

'When Chuck Norris is web surfing websites get the message "Warning: Internet Explorer has deemed this user to be malicious or dangerous. Proceed?".'

## Criando a Signature

A `Signature` define a interface entre nossa aplicação e o agente ReAct.

Teremos:

### Entrada

`pergunta`

Representa a solicitação feita pelo usuário.

### Saída

`resposta`

Representa a resposta final produzida pelo agente após utilizar,
quando necessário, uma ou mais ferramentas.

In [12]:
class AssistenteComFerramentas(dspy.Signature):
    """
    Responda à pergunta do usuário.

    Use as ferramentas disponíveis sempre que forem necessárias para
    obter dados financeiros ou uma piada/fato de Chuck Norris.
    """

    # Pergunta recebida pelo agente
    pergunta: str = dspy.InputField(
        desc="Pergunta ou solicitação feita pelo usuário"
    )

    # Resposta produzida depois da execução das ferramentas necessárias
    resposta: str = dspy.OutputField(
        desc=(
            "Resposta final em português, clara e baseada "
            "nos resultados das ferramentas"
        )
    )

## Criar Módulo ReAct

Agora criamos o agente utilizando `dspy.ReAct`.

Passamos três informações principais:

- `AssistenteComFerramentas`: Signature que define entrada e saída.
- `tools`: funções que o agente pode executar.
- `max_iters`: quantidade máxima de ciclos de raciocínio e uso de ferramentas.

O agente poderá escolher entre:

1. `consultar_acao`
2. `obter_piada_chuck_norris`

O DSPy também adiciona internamente uma ferramenta especial chamada
`finish`, utilizada pelo agente quando entende que já possui informações
suficientes para produzir a resposta.

In [13]:
agente = dspy.ReAct(
    AssistenteComFerramentas,  # Signature utilizada pelo agente

    tools=[
        consultar_acao,              # Tool baseada em yfinance
        obter_piada_chuck_norris     # Tool baseada na API chucknorris.io
    ],

    max_iters=6  # Máximo de ciclos de raciocínio/ação
)

In [14]:
resultado_financeiro = agente(
    pergunta=(
        "Qual foi a última cotação disponível da NVDA "
        "e qual foi sua variação percentual em relação "
        "ao pregão anterior?"
    )
)

resultado_financeiro

Prediction(
    trajectory={'thought_0': 'Vou consultar a última cotação disponível da ação NVDA usando a ferramenta de consulta de ações para obter o fechamento mais recente e a variação percentual em relação ao pregão anterior.', 'tool_name_0': 'consultar_acao', 'tool_args_0': {'ticker': 'NVDA'}, 'observation_0': '{"ticker": "NVDA", "data": "2026-09-04", "ultimo_fechamento": 230.36, "fechamento_anterior": 228.45, "variacao_percentual": 0.84}', 'thought_1': 'Tenho a informação necessária: a última cotação disponível da NVDA é de 230.36 (data: 2026-09-04), o fechamento anterior foi 228.45, logo a variação foi de +0.84% (alta de 1.91). Vou finalizar e retornar essa resposta ao usuário.', 'tool_name_1': 'finish', 'tool_args_1': {}, 'observation_1': 'Completed.'},
    reasoning='Usei a ferramenta de consulta de ações (consultar_acao). A resposta retornou os dados do ticker NVDA com data 2026-09-04, último fechamento 230,36, fechamento anterior 228,45 e variação percentual 0,84%. Interpret

## Ver Resposta Final

O campo `resposta` contém a resposta final construída pelo agente
depois de consultar as ferramentas necessárias.

In [15]:
print("=" * 70)
print("RESPOSTA FINAL")
print("=" * 70)

print(resultado_financeiro.resposta)

RESPOSTA FINAL
A última cotação disponível da NVDA (04/09/2026) foi US$ 230,36. Isso representa uma alta de +0,84% em relação ao pregão anterior (fechamento anterior US$ 228,45), um aumento absoluto de US$ 1,91.


## Executar ReAct - Chuck Norris

Agora fazemos uma solicitação que necessita apenas da API Chuck Norris.

O agente deverá identificar que a ferramenta financeira não é necessária
e selecionar `obter_piada_chuck_norris`.

In [16]:
resultado_piada = agente(
    pergunta=(
        "Conte uma piada de Chuck Norris "
        "relacionada à categoria dev."
    )
)

print(resultado_piada.resposta)

Piada (PT-BR): "Chuck Norris não precisa de garbage collection porque ele não chama .Dispose(), ele chama .DropKick()."

Original (EN): "Chuck Norris doesn't need garbage collection because he doesn't call .Dispose(), he calls .DropKick()."


## Executar ReAct com Múltiplas Tools

Nesta pergunta o agente precisa executar duas ações diferentes:

1. Consultar informações financeiras da NVIDIA através do `yfinance`.
2. Consultar uma piada através da API Chuck Norris.

O ReAct deverá perceber que nenhuma ferramenta isoladamente é suficiente
para responder à pergunta completa.

Consequentemente, ele poderá executar múltiplas ferramentas antes de
produzir a resposta final.

In [18]:
resultado = agente(
    pergunta=(
        "Consulte a última cotação disponível da NVDA e informe "
        "a variação percentual em relação ao pregão anterior. "
        "Depois conte uma piada de Chuck Norris da categoria dev."
    )
)

resultado

Prediction(
    trajectory={'thought_0': 'Vou primeiro consultar a última cotação disponível da NVDA para obter a variação percentual em relação ao pregão anterior. Depois, farei uma chamada para obter uma piada da categoria "dev".', 'tool_name_0': 'consultar_acao', 'tool_args_0': {'ticker': 'NVDA'}, 'observation_0': '{"ticker": "NVDA", "data": "2026-09-04", "ultimo_fechamento": 230.36, "fechamento_anterior": 228.45, "variacao_percentual": 0.84}', 'thought_1': 'Agora vou obter uma piada da categoria "dev" do Chuck Norris.', 'tool_name_1': 'obter_piada_chuck_norris', 'tool_args_1': {'categoria': 'dev'}, 'observation_1': "Chuck Norris doesn't go on the internet, he has every internet site stored in his memory. He refreshes webpages by blinking.", 'thought_2': 'Tenho as informações necessárias: cotação mais recente da NVDA (data, último fechamento, fechamento anterior e variação percentual) e uma piada da categoria "dev". Vou finalizar e apresentar a resposta ao usuário.', 'tool_name_2': 

## Ver Resposta Final

Agora exibimos somente a resposta final criada pelo agente após
executar as ferramentas necessárias.

In [20]:
print("=" * 70)
print("RESPOSTA FINAL")
print("=" * 70)

print(resultado.resposta)

RESPOSTA FINAL
Cotação NVDA (última disponível)
- Data: 2026-09-04
- Último fechamento: 230,36
- Fechamento anterior: 228,45
- Variação em relação ao pregão anterior: +0,84% (alta)

Piada de Chuck Norris — categoria "dev"
Chuck Norris não entra na internet; ele tem todos os sites armazenados em sua memória. Ele atualiza páginas apenas piscando.


## Inspecionando a Trajetória do ReAct

Uma das diferenças mais interessantes entre `ReAct` e módulos mais simples
é a existência de uma trajetória (`trajectory`).

A trajetória registra as sucessivas etapas executadas pelo agente,
incluindo:

- `thought_n`: etapa de decisão do agente;
- `tool_name_n`: ferramenta selecionada;
- `tool_args_n`: argumentos enviados à ferramenta;
- `observation_n`: valor retornado pela ferramenta.

Isso permite observar como o agente alterna entre decisões e ações antes
de produzir a resposta final.

In [21]:
print("=" * 70)
print("TRAJETÓRIA DO REACT")
print("=" * 70)

for chave, valor in resultado.trajectory.items():

    print(f"\n{chave}")
    print("-" * 70)
    print(valor)

TRAJETÓRIA DO REACT

thought_0
----------------------------------------------------------------------
Vou primeiro consultar a última cotação disponível da NVDA para obter a variação percentual em relação ao pregão anterior. Depois, farei uma chamada para obter uma piada da categoria "dev".

tool_name_0
----------------------------------------------------------------------
consultar_acao

tool_args_0
----------------------------------------------------------------------
{'ticker': 'NVDA'}

observation_0
----------------------------------------------------------------------
{"ticker": "NVDA", "data": "2026-09-04", "ultimo_fechamento": 230.36, "fechamento_anterior": 228.45, "variacao_percentual": 0.84}

thought_1
----------------------------------------------------------------------
Agora vou obter uma piada da categoria "dev" do Chuck Norris.

tool_name_1
----------------------------------------------------------------------
obter_piada_chuck_norris

tool_args_1
------------------------

## Ferramentas Utilizadas pelo Agente

A `trajectory` contém muitas informações.

Podemos filtrar apenas os campos `tool_name_*` para visualizar
rapidamente quais ferramentas foram escolhidas pelo ReAct.

In [22]:
print("=" * 70)
print("FERRAMENTAS UTILIZADAS")
print("=" * 70)

for chave, valor in resultado.trajectory.items():

    if chave.startswith("tool_name_"):

        print(f"{chave}: {valor}")

FERRAMENTAS UTILIZADAS
tool_name_0: consultar_acao
tool_name_1: obter_piada_chuck_norris
tool_name_2: finish


## Ações e Observações

Também podemos visualizar apenas a interação entre o agente e o ambiente.

Para cada ferramenta executada mostramos:

- nome da ferramenta;
- argumentos;
- observação retornada.

Isso torna mais fácil compreender o padrão **Action → Observation**
característico do ReAct.

In [23]:
trajetoria = resultado.trajectory

numero_etapas = len([
    chave
    for chave in trajetoria
    if chave.startswith("tool_name_")
])

for i in range(numero_etapas):

    print("=" * 70)
    print(f"ETAPA {i + 1}")
    print("=" * 70)

    print(
        "Tool:",
        trajetoria.get(f"tool_name_{i}")
    )

    print(
        "Args:",
        trajetoria.get(f"tool_args_{i}")
    )

    print(
        "Observation:",
        trajetoria.get(f"observation_{i}")
    )

    print()

ETAPA 1
Tool: consultar_acao
Args: {'ticker': 'NVDA'}
Observation: {"ticker": "NVDA", "data": "2026-09-04", "ultimo_fechamento": 230.36, "fechamento_anterior": 228.45, "variacao_percentual": 0.84}

ETAPA 2
Tool: obter_piada_chuck_norris
Args: {'categoria': 'dev'}
Observation: Chuck Norris doesn't go on the internet, he has every internet site stored in his memory. He refreshes webpages by blinking.

ETAPA 3
Tool: finish
Args: {}
Observation: Completed.



## Inspecionando o Histórico

Como `ReAct` pode realizar várias chamadas ao modelo durante uma única
execução, utilizamos `dspy.inspect_history()` para visualizar as chamadas
realizadas pelo DSPy.

Diferentemente de um `Predict` simples, uma execução ReAct normalmente
envolve diversas interações:

1. decidir qual ferramenta utilizar;
2. analisar a observação;
3. decidir a próxima ação;
4. finalizar;
5. gerar a resposta final.

Por isso utilizamos um valor maior de `n`.

In [24]:
# Exibe as últimas chamadas realizadas ao modelo
dspy.inspect_history(n=10)





[2026-09-07T11:36:16.420883]

System message:

Your input fields are:
1. `pergunta` (str): Pergunta ou solicitação feita pelo usuário
2. `trajectory` (str):
Your output fields are:
1. `next_thought` (str): 
2. `next_tool_name` (Literal['consultar_acao', 'obter_piada_chuck_norris', 'finish']): 
3. `next_tool_args` (dict[str, Any]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## pergunta ## ]]
{pergunta}

[[ ## trajectory ## ]]
{trajectory}

[[ ## next_thought ## ]]
{next_thought}

[[ ## next_tool_name ## ]]
{next_tool_name}        # note: the value you produce must exactly match (no extra characters) one of: consultar_acao; obter_piada_chuck_norris; finish

[[ ## next_tool_args ## ]]
{next_tool_args}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "additionalProperties": true}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Responda à pergunta do usuário